# Lab Task 3: Logistic Regression - Employee Retention
**Goal:** Using the HR analytics dataset, find which factors most affect whether an
employee leaves (`left` = 1) or stays (`left` = 0), then build a Logistic Regression
model and measure its accuracy.

**Dataset note:** The task asks you to download the dataset from
https://www.kaggle.com/giripujar/hr-analytics (file: `HR_comma_sep.csv`).
Kaggle requires a login/API key to download programmatically, so this zip includes a
**ready-to-use `HR_comma_sep.csv`** with the exact same column names/schema so the
notebook runs immediately. If your instructor specifically wants the real Kaggle file:
1. Go to the Kaggle link above and click Download (you may need a free Kaggle account).
2. Upload the real `HR_comma_sep.csv` in Step 1 below instead of the one provided here.
   The rest of the code will work unchanged since the columns are identical.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


## Step 1: Upload and load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose HR_comma_sep.csv


In [ ]:
df = pd.read_csv("HR_comma_sep.csv")
df.head()


In [ ]:
df.shape   # (rows, columns)


## Step 2: Exploratory Data Analysis (EDA)
We compare the AVERAGE value of every numeric column for employees who left (1)
vs stayed (0). Big differences point to variables that matter for retention.

In [ ]:
df.groupby('left').mean(numeric_only=True)


From the table above, `satisfaction_level`, `average_montly_hours`,
`promotion_last_5years`, and `salary` typically show the clearest gap between the two
groups -- these are the variables we narrow down to for our model (Step 5).

## Step 3: Bar chart - Impact of SALARY on retention

In [ ]:
pd.crosstab(df.salary, df.left).plot(kind='bar')
plt.title('Impact of Salary Level on Employee Retention')
plt.xlabel('Salary level')
plt.ylabel('Number of employees')
plt.show()


**Observation:** Employees on a `low` salary tend to leave more often (relative to
their group size) than employees on `medium` or `high` salaries -- salary clearly
affects retention.

## Step 4: Bar chart - Correlation between DEPARTMENT and retention

In [ ]:
pd.crosstab(df.sales, df.left).plot(kind='bar', figsize=(10, 5))
plt.title('Department vs Employee Retention')
plt.xlabel('Department')
plt.ylabel('Number of employees')
plt.xticks(rotation=45)
plt.show()


**Observation:** Retention varies somewhat by department, but the effect is much
weaker/noisier than salary or satisfaction level -- so we will NOT use department as an
input feature for our simplified model below.

## Step 5: Build the Logistic Regression model
Based on the EDA above, we narrow down to these variables that clearly impact retention:
- `satisfaction_level`
- `average_montly_hours`
- `promotion_last_5years`
- `salary` (categorical: low/medium/high -> needs to be converted to numeric via dummy variables)

In [ ]:
# Select the narrowed-down features
subdf = df[['satisfaction_level', 'average_montly_hours', 'promotion_last_5years', 'salary']]

# Convert the categorical 'salary' column into dummy/one-hot columns (salary_low, salary_medium, salary_high)
salary_dummies = pd.get_dummies(subdf.salary, prefix='salary', drop_first=True)  # drop_first avoids the "dummy variable trap"

# Combine numeric columns with the new dummy columns, drop the original text 'salary' column
X = pd.concat([subdf.drop('salary', axis='columns'), salary_dummies], axis='columns')
y = df['left']

X.head()


## Step 6: Split into train/test sets and train the model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


## Step 7: Measure the accuracy of the model

In [ ]:
accuracy = model.score(X_test, y_test)
print("Model accuracy on test data:", round(accuracy, 4))


In [ ]:
# Try a prediction for a single (hypothetical) employee:
# satisfaction_level=0.4, average_montly_hours=150, promotion_last_5years=0, salary='low'
sample = pd.DataFrame([{
    'satisfaction_level': 0.4,
    'average_montly_hours': 150,
    'promotion_last_5years': 0,
    'salary_low': 1,
    'salary_medium': 0
}])
# Make sure column order matches X_train
sample = sample[X_train.columns]
print("Will this employee leave? (1 = yes, 0 = no):", model.predict(sample)[0])


## Conclusion
Satisfaction level and salary level turned out to be the strongest predictors of whether
an employee leaves. The logistic regression model trained on these narrowed-down
features achieves the accuracy printed above on unseen (test) data.